# Week 3 · Lab: Text classification

Companion to [Text Classification](https://revanthreddy-hai.github.io/the-llm-residency/week03.html).
Every number in the essay, reproduced end to end: the four rows, the normalization ablation, the
paired McNemar test, the ten-seed label curves, and the zero-shot margin AUC. The fine-tuning cell
takes about ten minutes on a free Colab CPU (under a minute on a GPU runtime); everything else runs
in seconds.

*Self-checks are `assert` cells. If one fails, something real changed — read it before moving on.*


## 1 · Set up the environment


In [ ]:
# Colab and fresh environments install the pinned dependencies on first run.
# Where they are already present, this cell just confirms the imports resolve.
# If versions differ (e.g. stock Colab), pip fetches the pinned wheels; on GPU
# runtimes the torch wheel alone can be a few GB, and a runtime restart may be
# needed before the new version is actually imported.
from importlib.metadata import version, PackageNotFoundError

DEPENDENCIES = {
    "torch": "torch==2.13.0",
    "transformers": "transformers==5.16.1",
    "sentence_transformers": "sentence-transformers==6.0.1",
    "scikit_learn": "scikit-learn==1.9.0",
    "datasets": "datasets==5.0.1",
}

def installed(pip_spec):
    name, pinned = pip_spec.split("==")
    try:
        return version(name) == pinned
    except PackageNotFoundError:
        return False

mismatched = [spec for spec in DEPENDENCIES.values() if not installed(spec)]
if mismatched:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *mismatched])

import torch, transformers, sklearn
SEED = 0
torch.manual_seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

# every load below pins the exact artifact the essay measured
GPT2_REV = "607a30d783dfa663caf39e06633721c8d4cfcd7e"
MINILM = "sentence-transformers/all-MiniLM-L6-v2"
MINILM_REV = "1110a24"
DATASET = "cornell-movie-review-data/rotten_tomatoes"
DATASET_REV = "aa13bc287fa6fcab6daf52f0dfb9994269ffea28"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device", DEV)


## 2 · The task: 8,530 labeled reviews, 1,066 held out

Pang and Lee's 2005 sentence-polarity corpus in the Hugging Face 80/10/10 packaging. The
1,066-sentence validation split goes untouched: nothing in this lab is tuned, every knob is a
library default.


In [ ]:
import numpy as np
from datasets import load_dataset

ds = load_dataset(DATASET, revision=DATASET_REV)
train_texts, train_y = list(ds["train"]["text"]), np.array(ds["train"]["label"])
test_texts, test_y = list(ds["test"]["text"]), np.array(ds["test"]["label"])
print(len(train_texts), "train /", len(test_texts), "test /", len(ds["validation"]), "validation (unused)")
print("example:", repr(test_texts[0]), "| label", test_y[0])


In [ ]:
# self-check: sizes and balance -- always-positive scores exactly 0.500 on this test set
assert len(train_texts) == 8530 and len(test_texts) == 1066
assert abs(float(train_y.mean()) - 0.5) < 1e-9 and abs(float(test_y.mean()) - 0.5) < 1e-9
print("ok — both splits exactly balanced")


## 3 · The reader, frozen: embeddings + a 385-parameter head


In [ ]:
import time
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

t0 = time.perf_counter()
reader = SentenceTransformer(MINILM, device=DEV, revision=MINILM_REV)
Xtr_r = reader.encode(train_texts, batch_size=128, show_progress_bar=False)
Xte_r = reader.encode(test_texts, batch_size=128, show_progress_bar=False)
t_embed_reader = time.perf_counter() - t0

def head_fit(Xtr, ytr):
    return LogisticRegression(max_iter=2000, random_state=SEED).fit(Xtr, ytr)

pred_r = head_fit(Xtr_r, train_y).predict(Xte_r)
acc_r = round(float(accuracy_score(test_y, pred_r)), 4)
f1_r = round(float(f1_score(test_y, pred_r)), 4)
print(f"reader frozen: acc {acc_r} · f1 {f1_r} · embed {t_embed_reader:.1f}s · {Xtr_r.shape[1] + 1} trained params")


In [ ]:
# self-check: the reader's frozen features land where the essay measured (0.774 on MPS)
assert Xtr_r.shape == (8530, 384)
assert 0.75 < acc_r < 0.80, acc_r
print("ok — reader frozen at", acc_r)


## 4 · The writer, frozen: GPT-2 mean-pooled + the same recipe

Plus the ablation that nearly fooled the essay: the reader's pipeline L2-normalizes its vectors by
design, the writer's does not. Normalize the writer's the same way and watch.


In [ ]:
from transformers import AutoTokenizer, AutoModel

tok_g = AutoTokenizer.from_pretrained("gpt2", revision=GPT2_REV)
tok_g.pad_token = tok_g.eos_token
writer = AutoModel.from_pretrained("gpt2", revision=GPT2_REV).to(DEV).eval()

def writer_embed(texts, bs=64):
    vecs = []
    with torch.no_grad():
        for i in range(0, len(texts), bs):
            enc = tok_g(texts[i:i+bs], return_tensors="pt", padding=True,
                        truncation=True, max_length=128).to(DEV)
            h = writer(**enc).last_hidden_state           # [B, T, 768]
            mask = enc["attention_mask"].unsqueeze(-1)
            vecs.append(((h * mask).sum(1) / mask.sum(1)).float().cpu().numpy())
    return np.concatenate(vecs)

t0 = time.perf_counter()
Xtr_w = writer_embed(train_texts); Xte_w = writer_embed(test_texts)
t_embed_writer = time.perf_counter() - t0

pred_w = head_fit(Xtr_w, train_y).predict(Xte_w)
acc_w = round(float(accuracy_score(test_y, pred_w)), 4)
f1_w = round(float(f1_score(test_y, pred_w)), 4)

def l2norm(X): return X / np.linalg.norm(X, axis=1, keepdims=True)
acc_wn = round(float(accuracy_score(
    test_y, head_fit(l2norm(Xtr_w), train_y).predict(l2norm(Xte_w)))), 4)
print(f"writer frozen: acc {acc_w} · f1 {f1_w} · embed {t_embed_writer:.1f}s")
print(f"writer frozen, L2-normalized: acc {acc_wn}  <- one preprocessing flag")


In [ ]:
# self-check: raw beats the reader here, and normalization costs the writer ~20 points
assert Xtr_w.shape == (8530, 768)
assert acc_w > acc_r + 0.01, (acc_w, acc_r)          # the essay's 0.800 vs 0.774
assert acc_w - acc_wn > 0.15, (acc_w, acc_wn)        # the 20-point pipeline lever
print("ok — writer raw", acc_w, "vs normalized", acc_wn, "vs reader", acc_r)


## 5 · Is 2.6 points a real gap? McNemar's paired test

Accuracy differences on a shared test set deserve a paired test: count the reviews where exactly
one model is right, and ask whether the split of those disagreements could be chance.


In [ ]:
from scipy import stats as sps

reader_only = int(((pred_r == test_y) & (pred_w != test_y)).sum())
writer_only = int(((pred_r != test_y) & (pred_w == test_y)).sum())
p_mcnemar = round(float(sps.binomtest(min(reader_only, writer_only),
                                      reader_only + writer_only, 0.5).pvalue), 4)
print(f"disagreements: reader-only-right {reader_only}, writer-only-right {writer_only}, p = {p_mcnemar}")


In [ ]:
# self-check: the gap is suggestive, not settled (essay: p = 0.08)
assert writer_only > reader_only, (writer_only, reader_only)
assert 0.01 < p_mcnemar < 0.30, p_mcnemar
print("ok — p =", p_mcnemar, ": don't crown a winner on 1,066 reviews")


## 6 · Accuracy against labels, ten seeds per point

The essay's first draft claimed the reader wins when labels are scarce, on the strength of one
random subset. Ten seeds per budget tell a different story: overlapping bands, writer's mean ahead.


In [ ]:
SIZES = [50, 200, 1000]
curves = {}
for name, Xtr, Xte in [("reader", Xtr_r, Xte_r), ("writer", Xtr_w, Xte_w)]:
    rows = []
    for n in SIZES:
        accs = []
        for s in range(10):
            rng = np.random.RandomState(1000 + s)
            sub = np.sort(rng.permutation(len(train_texts))[:n])
            accs.append(accuracy_score(test_y, head_fit(Xtr[sub], train_y[sub]).predict(Xte)))
        rows.append((n, round(float(np.mean(accs)), 4),
                     round(float(np.min(accs)), 4), round(float(np.max(accs)), 4)))
    curves[name] = rows
    print(name, "  n | mean [min–max]")
    for n, m, lo, hi in rows:
        print(f"  {n:>5} | {m:.4f} [{lo:.4f}–{hi:.4f}]")


In [ ]:
# self-check: no low-data crossover survives averaging -- writer's mean >= reader's at every size,
# and the 50-label bands overlap heavily (which subset you draw matters more than which model)
for (n, mr, lor, hir), (_, mw, low, hiw) in zip(curves["reader"], curves["writer"]):
    assert mw >= mr - 0.005, (n, mr, mw)
r50, w50 = curves["reader"][0], curves["writer"][0]
assert max(r50[2], w50[2]) < min(r50[3], w50[3]), (r50, w50)   # bands overlap at n=50
print("ok — one-seed curves lie; ten-seed means put the writer ahead throughout")


## 7 · The reader, taught: fine-tune all 22.7M parameters

The slow cell: two epochs over 8,530 reviews. Roughly ten minutes on a free Colab CPU, under a
minute on a GPU runtime, about half a minute on Apple-silicon MPS.


In [ ]:
from transformers import AutoModelForSequenceClassification

tok_m = AutoTokenizer.from_pretrained(MINILM, revision=MINILM_REV)
clf = AutoModelForSequenceClassification.from_pretrained(
    MINILM, revision=MINILM_REV, num_labels=2).to(DEV)
opt = torch.optim.AdamW(clf.parameters(), lr=2e-5)
BS, EPOCHS = 32, 2

t0 = time.perf_counter()
clf.train()
order = np.arange(len(train_texts))
for ep in range(EPOCHS):
    ep_rng = np.random.RandomState(SEED + ep); ep_rng.shuffle(order)
    for i in range(0, len(order), BS):
        b = order[i:i+BS]
        enc = tok_m([train_texts[j] for j in b], return_tensors="pt", padding=True,
                    truncation=True, max_length=128).to(DEV)
        loss = clf(**enc, labels=torch.tensor(train_y[b]).to(DEV)).loss
        loss.backward(); opt.step(); opt.zero_grad()
t_train = time.perf_counter() - t0

clf.eval()
preds = []
with torch.no_grad():
    for i in range(0, len(test_texts), 128):
        enc = tok_m(test_texts[i:i+128], return_tensors="pt", padding=True,
                    truncation=True, max_length=128).to(DEV)
        preds.append(clf(**enc).logits.argmax(-1).cpu().numpy())
pred_ft = np.concatenate(preds)
acc_ft = round(float(accuracy_score(test_y, pred_ft)), 4)
f1_ft = round(float(f1_score(test_y, pred_ft)), 4)
print(f"reader fine-tuned: acc {acc_ft} · f1 {f1_ft} · train {t_train:.1f}s")


In [ ]:
# self-check: teaching the reader beats every frozen row (0.827 as measured; fine-tuning
# is not bit-deterministic across devices, so bounds rather than equality)
assert acc_ft >= 0.81, acc_ft
assert acc_ft > acc_w + 0.005 and acc_ft > acc_r + 0.02, (acc_ft, acc_w, acc_r)
print("ok — fine-tuned reader at", acc_ft)


## 8 · The writer, writing: zero-shot prompting, then the margin

Classification by continuation, made precise: nothing is generated. For each review, compare the
scores the model assigns to exactly two candidate tokens after the prompt, and take the larger.
Then the measurement the essay almost skipped: rank the reviews by the score *margin* and ask
whether the ordering knows the classes, regardless of where zero sits.


In [ ]:
from transformers import AutoModelForCausalLM
from sklearn.metrics import roc_auc_score

lm = AutoModelForCausalLM.from_pretrained("gpt2", revision=GPT2_REV).to(DEV).eval()
id_pos = tok_g.encode(" positive")[0]; id_neg = tok_g.encode(" negative")[0]

def zeroshot(template):
    preds, margins = [], []
    suffix_ids = tok_g.encode(template.format(t="")[template.format(t="").find("\n"):])
    with torch.no_grad():
        for i in range(0, len(test_texts), 64):
            batch = [template.format(t=t) for t in test_texts[i:i+64]]
            enc = tok_g(batch, return_tensors="pt", padding=True,
                        truncation=True, max_length=160).to(DEV)
            # the suffix must survive truncation, or we'd be scoring review continuations
            lens = enc["attention_mask"].sum(1)
            assert int(lens.max()) < 160, "truncation would clip the question suffix"
            logits = lm(**enc).logits
            last = lens - 1
            row = logits[torch.arange(len(batch)), last]
            preds.append((row[:, id_pos] > row[:, id_neg]).long().cpu().numpy())
            margins.append((row[:, id_pos] - row[:, id_neg]).float().cpu().numpy())
    p = np.concatenate(preds); m = np.concatenate(margins)
    return round(float(accuracy_score(test_y, p)), 4), round(float(p.mean()), 4), m

acc_z1, frac1, margins = zeroshot("Review: {t}\nThe sentiment of this review is")
acc_z2, frac2, _ = zeroshot("{t}\nQuestion: Is the review above positive or negative?\nAnswer: It is")
auc = round(float(roc_auc_score(test_y, margins)), 4)
acc_med = round(float(accuracy_score(test_y, (margins > np.median(margins)).astype(int))), 4)
print(f"prompt 1: acc {acc_z1} · said 'positive' for {frac1:.1%} of reviews")
print(f"prompt 2: acc {acc_z2} · said 'positive' for {frac2:.1%} of reviews")
print(f"margin AUC {auc} · accuracy {acc_med} if the cut sits at the median margin instead of zero")


In [ ]:
# self-check: collapse at the interface, signal in the margins (essay: 0.502 / 99.8% / AUC 0.797)
assert acc_z1 < 0.55 and acc_z2 < 0.55, (acc_z1, acc_z2)
assert frac1 > 0.95 and frac2 > 0.95, (frac1, frac2)
assert auc > 0.75, auc
print("ok — broken zero point, present knowledge")


## What you just did

- Ran the frozen recipe on two checkpoints: the writer's raw features won, 0.800 against 0.774,
  and a paired McNemar test filed that gap under suggestive (p = 0.08), not settled.
- Found a 20-point pipeline lever: L2-normalizing the writer's vectors, which the reader's
  pipeline does by design, collapses the writer to 0.604. Preprocessing is part of the model.
- Replaced a one-seed label curve with ten seeds per point and watched a low-data “crossover”
  evaporate into overlapping bands.
- Fine-tuned the 22.7M-parameter reader past both frozen rows (0.827) in the time a coffee takes.
- Measured a prompting collapse precisely: the two-token readout says “positive” for essentially
  every review (0.502), while the same forward pass's score margin ranks the classes at AUC 0.797.
  The knowledge is there; the interface's zero point is broken.

Next week prompting starts working — on a model trained to treat text as instructions — and the
question becomes which parts of a prompt actually move accuracy, measured against an eval set.
